In [1]:
# Check core SDK version number
import azureml.core
print("SDK version:", azureml.core.VERSION)


SDK version: 1.51.0


In [2]:
from azureml.core import Workspace

ws = Workspace.from_config()
print(ws.name, ws.resource_group, ws.location, ws.subscription_id, sep='\n')


mlv2
ml_group
eastasia
956a5d16-ed62-4076-890a-4f7ea58eef95


In [6]:
from azureml.core.model import Model
# Tip: When model_path is set to a directory, you can use the child_paths parameter to include
#      only some of the files from the directory
model = Model.register(model_path = "raman_pls_model.pkl",
                       model_name = "raman_pls_model",
                       workspace = ws)




Registering model raman_pls_model


In [3]:
from azureml.core.model import InferenceConfig
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies

# Create the environment
myenv = Environment(name="myenv_raman4")
conda_dep = CondaDependencies()

# Define the packages needed by the model and scripts
# conda_dep.add_conda_package("keras==2.11.0")
conda_dep.add_conda_package("numpy==1.24.2")
conda_dep.add_conda_package("pandas==1.5.3")
# You must list azureml-defaults as a pip dependency
conda_dep.add_pip_package("scikit-learn==1.3.0")
# conda_dep.add_pip_package("pickle")

conda_dep.add_pip_package("scipy==1.10.1")
# conda_dep.add_pip_package("tensorflow==2.11.0")


# Adds dependencies to PythonSection of myenv
myenv.python.conda_dependencies=conda_dep

inference_config = InferenceConfig(entry_script="score_raman_2.py",
                                   environment=myenv)



In [7]:
from azureml.core.webservice import AksWebservice, Webservice
from azureml.core.model import Model
from azureml.core.compute.aks import AksCompute
from azureml.core.compute import ComputeTarget

aks_target = AksCompute(ws,"dev-aks2-v1")
deployment_config = AksWebservice.deploy_configuration(cpu_cores = 1, memory_gb = 1)

network = Model(ws, name='raman_pls_model')

service = Model.deploy(ws, 'raman-pls-service-v2-5', [network], inference_config, deployment_config, aks_target)

service.wait_for_deployment(show_output = True)
print(service.state)
print(service.get_logs())


/tmp/ipykernel_13655/1740615119.py:11: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(ws, 'raman-pls-service-v2-5', [network], inference_config, deployment_config, aks_target)


Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2024-04-13 17:27:16+00:00 Creating Container Registry if not exists.
2024-04-13 17:27:16+00:00 Registering the environment.
2024-04-13 17:27:17+00:00 Use the existing image.
2024-04-13 17:27:18+00:00 Creating resources in AKS.
2024-04-13 17:27:19+00:00 Submitting deployment to compute.
2024-04-13 17:27:19+00:00 Checking the status of deployment raman-pls-service-v2-5.

In [11]:
print(service.get_logs())


Received bad response from Model Management Service:
Response Code: 404
Headers: {'Date': 'Thu, 11 Apr 2024 05:05:01 GMT', 'Content-Type': 'application/json', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Vary': 'Accept-Encoding', 'x-ms-client-request-id': '9a3865cc-a32c-4099-955b-bfb0abdfad40', 'x-ms-client-session-id': '3c546a9a-21d3-4b2d-b2e3-8210823bde16', 'api-supported-versions': '1.0, 2018-03-01-preview, 2018-11-19', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'X-Content-Type-Options': 'nosniff', 'x-aml-cluster': 'vienna-eastasia-02', 'x-request-time': '0.025', 'Content-Encoding': 'gzip'}
Content: b'{"code":"NotFound","statusCode":404,"message":"The specified resource was not found.","details":[{"code":"NoSuchService","message":"There is no service with name: raman-pls-service-v2-5 in Subscription: 956a5d16-ed62-4076-890a-4f7ea58eef95, ResourceGroup: ml_group, Workspace: mlv2, ACR: /subscriptions/956a5d16-ed62-4076-890a-4f7ea58eef9

WebserviceException: WebserviceException:
	Message: Received bad response from Model Management Service:
Response Code: 404
Headers: {'Date': 'Thu, 11 Apr 2024 05:05:01 GMT', 'Content-Type': 'application/json', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Vary': 'Accept-Encoding', 'x-ms-client-request-id': '9a3865cc-a32c-4099-955b-bfb0abdfad40', 'x-ms-client-session-id': '3c546a9a-21d3-4b2d-b2e3-8210823bde16', 'api-supported-versions': '1.0, 2018-03-01-preview, 2018-11-19', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'X-Content-Type-Options': 'nosniff', 'x-aml-cluster': 'vienna-eastasia-02', 'x-request-time': '0.025', 'Content-Encoding': 'gzip'}
Content: b'{"code":"NotFound","statusCode":404,"message":"The specified resource was not found.","details":[{"code":"NoSuchService","message":"There is no service with name: raman-pls-service-v2-5 in Subscription: 956a5d16-ed62-4076-890a-4f7ea58eef95, ResourceGroup: ml_group, Workspace: mlv2, ACR